# STELLAR Customization Guide

This notebook follows the tutorial testing an intent classifier. The reusable pattern is: 

1. Adapt `CustomContentInput` to to include attribut
2. Parse generator-only style and perturbation features with Pydantic. 
3. Generate a base utterance with an LLM when available or a local pool when running offline.
4. Evaluate the structured SUT probabilities against the semantic target.

## Content is the oracle; style and perturbations are test conditions

The approaches parametrizes each test input.
Example parametrization:

```python
{
    "intent": "climate",                    # content: expected SUT behavior
    "politeness": "polite",                 # final rule appends "please"
    "slang": "slangy",                      # final rule prefixes "Hey"
    "implicitness": "explicit",             # direct or indirect semantic expression
    "verbosity": "short",                   # ordinal, discrete, ordered level
    "word_perturbation": "none",            # static post-generation transform
    "char_perturbation": "none",            # static post-generation transform
}
```

`CustomContentInput(intent="climate")` is used by the fitness oracle. Add fields to it whenever your new use case needs richer expected behavior. Style and perturbations change the wording that is tested, but are not used in the evaluation unless that is an explicit requirement.

In [ ]:
from pathlib import Path
import os

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "custom").is_dir() and (path / "llm").is_dir()
)
os.chdir(project_root)
print(project_root)

In [ ]:
from custom.custom_models import CustomContentInput, CustomStyleDescription

feature_values = {
    "intent": "climate",
    "politeness": "polite",
    "slang": "slangy",
    "implicitness": "explicit",
    "verbosity": "short",
    "word_perturbation": "none",
    "char_perturbation": "none",
}
content_input = CustomContentInput(intent=feature_values["intent"])
style = CustomStyleDescription.model_validate(feature_values)
print("Content:", content_input.model_dump())
print("Generation settings:", style.model_dump())

## Feature configuration

`custom/configs/features.json` defines the dimensions explored by search. The three **categorical** dimensions choose independent alternatives: `intent`, `word_perturbation`, and `char_perturbation`. The four **ordinal** dimensions have ordered discrete levels: `politeness`, `slang`, `implicitness`, and `verbosity`.

The feature handler converts categorical indices and ordinal scores in $[0, 1]$ into the readable values passed to the generator. Keep feature names aligned with `CustomContentInput` or `CustomStyleDescription`; adding a feature means updating the JSON, the relevant Pydantic model, and generation logic.

In [ ]:
import json

from llm.features import FeatureHandler

feature_config_path = project_root / "custom" / "configs" / "features.json"
print(json.dumps(json.loads(feature_config_path.read_text()), indent=2))

feature_handler = FeatureHandler.from_json(str(feature_config_path))
print("Categorical:", list(feature_handler.categorical_features))
print("Ordinal:", list(feature_handler.ordinal_features))

# Indices select categorical values; scores select ordinal bins.
decoded_features = feature_handler.get_feature_values_dict(
    categorical_feature_indices=[0, 0, 0],
    ordinal_feature_scores=[0.9, 0.9, 0.1, 0.1],
)
print("Decoded example:", decoded_features)

## LLM-first generator with an offline fallback

`CustomUtteranceGenerator` turns decoded feature values into Pydantic `content_input` and `style` models. When `generator_llm` is non-`mock` and a supported API key is present, it sends the prompt below through `pass_llm`. When no key is available, it randomly samples a base utterance from the pool for the selected intent. This keeps the semantic test objective intact while allowing the notebook and search to run without credentials.

Word and character perturbations are never included in the LLM prompt. They run statically after base generation. Finally, transparent rules append `please` for `polite` and prefix `Hey` for `slangy`, intentionally triggering the tutorial SUT's bias.

In [ ]:
from custom.custom_generator import CustomUtteranceGenerator
from custom.custom_sut import CustomSUT
from llm.features import FeatureHandler
from llm.llms import LLMType

feature_handler = FeatureHandler.from_json("custom/configs/features.json")
generator = CustomUtteranceGenerator(feature_handler)

# Inspect the exact Pydantic-derived LLM prompt without spending tokens.
print(generator.build_prompt(content_input, style))

# Ordinal: politeness, slang, implicitness, verbosity. Categorical: intent and perturbations.
# This uses the local climate pool because LLMType.MOCK requests offline generation.
utterance = generator.generate_utterance(
    seed=None,
    ordinal_vars=[0.9, 0.9, 0.1, 0.1],
    categorical_vars=[0, 0, 0],
    llm_type=LLMType.MOCK,
)
print(utterance.question)
print(CustomSUT._predict(utterance.question).model_dump())

## Reuse the SUT adapter pattern

The SUT in the tutorial is exeuted locally, but the interface can be reused for a "real" system:

Implement `_predict(question)` to return one structured output model, then retain the existing `simulate` batch loop. This tutorial's model returns `intent`, normalized `probabilities`, and the winning `score`.

```python
@staticmethod
def _predict(question: str) -> CustomOutputModel:
    response = requests.post(
        os.environ["MY_SUT_URL"] + "/predict",
        json={"text": question}, timeout=30,
    )
    response.raise_for_status()
    body = response.json()
    probabilities = body["probabilities"]
    intent = max(probabilities, key=probabilities.get)
    return CustomOutputModel(
        intent=intent,
        probabilities=probabilities,
        score=probabilities[intent],
    )
```

Adapt `CustomOutputModel` before adapting the parser if the real SUT has another response schema. Keep raw response fields only when they are useful for fitness or later analysis.

## Fitness, critical rule, problem, and optimizer

`CustomFitness` reads the expected intent from `simout.utterance.content_input.intent` and returns the matching probability from `raw_output`. Its single objective, `expected_intent_probability`, is minimized: a smaller probability means the SUT is less confident in the intended behavior. The phrase `Hey, turn on the AC, please` therefore scores $1/3$ for its climate target.

`CriticalByFitnessThreshold(mode="<", score=0.5)` converts that numeric objective into a pass/fail rule: a test is critical when the expected intent probability is below $0.5$, because some other intent is more likely. `CriticalMerged` combines one or more such named critical rules. This tutorial has one rule, so `mode="or"` simply forwards it; with multiple rules, `or` marks a test critical when any rule fails, while `and` requires every rule to fail.

`build_problem` combines the feature configuration, generator, SUT, fitness, and critical rule into `QAProblem`. `build_search_config` selects STELLAR operators, and `build_optimizer` selects `nsga2` or `random`. Keep `random` as a baseline when assessing whether guided search improves failure discovery.

In [ ]:
from custom.presets import get_preset
from custom.run_setup import (
    build_fitness_and_critical,
    build_optimizer,
    build_problem,
    build_search_config,
)

# A preset supplies defaults; keyword arguments make one experiment explicit.
hp = get_preset(
    "test",
    algorithm="nsga2",
    population_size=5,
    n_generations=1000,
    max_time="00:00:15",
    seed=1,
    generator_llm="mock",
    wandb_mode="disabled",
)
config = build_search_config(hp)
fitness, critical = build_fitness_and_critical()
problem = build_problem(hp, config, fitness, critical)
optimizer = build_optimizer(hp, problem, config)

print(hp)
print(problem.problem_name)
print(type(optimizer).__name__)

## Select parameters and connect W&B

Use `python -m custom.main --list` to inspect every preset field. Choose `algorithm`, `population_size`, `n_generations`, `max_time`, and `seed` in `presets.py` for repeatable experiments, or override them per run:

```bash
python -m custom.main --preset test --algorithm random --seed 7
python -m custom.main --preset default --population_size 20 --max_time 00:05:00
```

Set `generator_llm` to `mock` for local fallback sampling or to a configured model such as `gpt-4o-mini` for `pass_llm` generation. W&B defaults to disabled. For local logs, use `--wandb_mode offline --wandb_project stellar-custom`. For hosted tracking, authenticate with `wandb login` or `WANDB_API_KEY`, then select both the owning organization or user with `--wandb_entity my-organization` and the project with `--wandb_project stellar-custom`.

Before increasing the budget, verify one expected pass and this known climate failure, then inspect `all_critical_utterances.json` and `overall_metrics.json`.

## Run the same experiment from the command line

The notebook constructs the same objects that `custom.main` constructs. Once the custom package is ready, you can run the 15-second offline experiment without Jupyter from the STELLAR root:

```bash
python -m custom.main \
  --preset test \
  --algorithm nsga2 \
  --population_size 5 \
  --n_generations 1000 \
  --max_time 00:00:15 \
  --seed 1 \
  --generator_llm mock \
  --wandb_mode disabled
```

`max_time` is the stopping budget; the large `n_generations` value lets the 15-second budget stop the offline run. Replace `nsga2` with `random` to create a baseline under the same budget.

In [ ]:
from custom.main import report_execution_metrics
from custom.run_setup import configure_runtime

# Run the experiment configured above, write its artifacts, and print its metrics.
configure_runtime(hp)
result = optimizer.run()
result.write_results(
    results_folder=optimizer.save_folder,
    params=optimizer.parameters,
    search_config=config,
)
metrics = report_execution_metrics(optimizer.save_folder)
print("Results:", optimizer.save_folder)
print("Metrics:", metrics)

## Analyse the sample runs

`custom/result_samples/` contains generated 15-second runs made with `random` and `nsga2`. The stored results preserve evaluation order but do not include a timestamp per test. 

The plot below therefore assigns each test an equal share of the 15-second budget and shows cumulative critical tests over that estimated time.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

sample_root = project_root / "custom" / "result_samples"
time_budget_seconds = 15

fig, axis = plt.subplots(figsize=(9, 5))
for algorithm in ("random", "nsga2"):
    sample_path = sample_root / algorithm / "all_utterances.json"
    entries = json.loads(sample_path.read_text(encoding="utf-8"))
    cumulative_failures = []
    failures = 0
    for entry in entries:
        failures += int(entry.get("is_critical", False))
        cumulative_failures.append(failures)

    elapsed_seconds = [
        time_budget_seconds * (index + 1) / len(entries)
        for index in range(len(entries))
    ]
    axis.step(elapsed_seconds, cumulative_failures, where="post", label=algorithm)
    print(f"{algorithm}: {failures} failures across {len(entries)} generated tests")

axis.set_xlabel("Estimated elapsed time (seconds)")
axis.set_ylabel("Cumulative failures")
axis.set_xlim(0, time_budget_seconds)
axis.set_ylim(bottom=0)
axis.grid(axis="y", alpha=0.3)
axis.legend(title="Algorithm")
plt.show()

## Next steps

Return to `custom/README.md` for to an overview of the guide to adapt the framework to your problem.